# Optimization of a three-sphere swimmer

### Imports

In [ ]:
import numpy as np
import jax.numpy as jnp
import jax
import optax
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import softmobility as sm

np.set_printoptions(precision=3, linewidth=100, suppress=True, sign=" ")

## Parameters

In [ ]:
# Active forcing: F = AMP_F * sin(t)
AMP_F = 10     

# Scalar active force
myscalar = sm.Scalar(
    lambda pos, t, p: p[0] * jnp.sin(t),
    params=[AMP_F,],
    param_names=["amp_force",],
)      

In [ ]:
# Create a quiet flow 
no_flow = sm.no_flow()

In [ ]:
# numerical simulation parameters
FINAL_TIME = 24 * jnp.pi    # final time of simulations
N_DT = 64                   # number of time steps per period

# Optimization parameters
N_TRANSIENT = 32            # number of periods before optimization begins

# Calculated paramters
DT = 2.0 * np.pi / N_DT
N_STEPS = int(FINAL_TIME / DT)

### YAML file

In [ ]:
yaml_data = """
# Variable Prefixes (Optional)
design_names:      
  - spring_k    
  - radius
  - distance
input_names:
  - active_force
    
# Default Values (Optional)
defaults:
  _ref_: 1
  spring_k: .5
  distance0: 0.5
  distance1: 0.5
  radius1: 0.5
  radius2: 0.5
  _spr_: 1
  _rad_: 10
  _len_: 10
  
# Spheres (Mandatory)
spheres:
  - # Sphere center #################
    radius: _ref_                    
    force: [-active_force - _spr_ * spring_k * dof0, 0, 0]
      
  - # Sphere left ###################
    radius: _rad_ * radius1               
    position: [-(_ref_ + _rad_ * radius1 + _len_ * distance0 + dof0) , 0, 0]
    force: [_spr_ * spring_k * dof0, 0, 0]
    
  - # Sphere right ##################
    radius: _rad_ * radius2
    position: [_ref_ + _rad_ * radius2 + _len_ * distance1 + dof1 , 0, 0]
    force: [active_force, 0, 0]
"""

## Simulation of default parameters

### Soft Body

In [ ]:
mybody= sm.SoftBody(yaml_data, verbose=True)
print(mybody.design_variables)

### Flow-body solver

In [ ]:
ROLLOUT = sm.FlowBodyRollout(
    soft_body=mybody, 
    flow=no_flow, 
    input_map={"active_force": myscalar}
)

In [ ]:
positions, orientations, dofs = ROLLOUT.rollout(DT, N_STEPS)

### Function for figure 

In [ ]:
def make_figure(positions, orientations, dofs):
    # Time vector
    time = jnp.linspace(0, FINAL_TIME, N_STEPS)

    # Create a subplot figure with 3 rows and 1 column
    fig = make_subplots(
        rows=3, cols=1,
        subplot_titles=("DOF", "Position (X, Y, Z)", "Orientation (X, Y, Z)"),
        shared_xaxes=True,  # Share the x-axis for all plots (time)
        vertical_spacing=0.1  # Adjust vertical spacing (reduce clutter)
    )

    # Plot DOFs in the first subplot 
    for i in range(mybody.Ndof):
        fig.add_trace(go.Scatter(x=time, y=dofs[:, i], mode='lines', name='DOF ' + str(i)), row=1, col=1)

    # Plot position components (X, Y, Z) in the second subplot
    fig.add_trace(go.Scatter(x=time, y=positions[:, 0], mode='lines', name='Position X'), row=2, col=1)
    fig.add_trace(go.Scatter(x=time, y=positions[:, 1], mode='lines', name='Position Y'), row=2, col=1)
    fig.add_trace(go.Scatter(x=time, y=positions[:, 2], mode='lines', name='Position Z'), row=2, col=1)

    # Plot orientation components (X, Y, Z) in the third subplot
    fig.add_trace(go.Scatter(x=time, y=orientations[:, 0], mode='lines', name='Orientation X'), row=3, col=1)
    fig.add_trace(go.Scatter(x=time, y=orientations[:, 1], mode='lines', name='Orientation Y'), row=3, col=1)
    fig.add_trace(go.Scatter(x=time, y=orientations[:, 2], mode='lines', name='Orientation Z'), row=3, col=1)

    # Update layout for titles and labels
    fig.update_layout(
        title="Trajectory Data Over Time",
        xaxis_title="Time (t)",
        # yaxis_title="Values (Position/Orientation Components)",
        template="plotly_white",  # Set the background to white
        showlegend=True,  # Show legend to distinguish between the traces
        height=800,  # Set figure height
        width=800,   # Set figure width
        plot_bgcolor='white',  # Set the plot background to white
        paper_bgcolor='white'  # Set the paper background to white
    )
    return fig

### Figure

In [ ]:
fig = make_figure(positions, orientations, dofs)
fig.show()

## Numerical optimization

### Objective

In [ ]:
def myobjective(myrollout, design):
    # Run N_TRANSIENT periods
    positions, orientations, dofs = myrollout.rollout(        
        dt=DT, 
        n_steps=N_DT * N_TRANSIENT,
        design=design)
    
    # Run one period for evaluation of mean velocity along x
    x_init = positions[-1,0]   # capture before evaluation
    positions, orientations, dofs = myrollout.rollout(
        dt=DT, 
        n_steps=N_DT,
        init_position=positions[-1],
        init_orientation=orientations[-1],
        init_dofs=dofs[-1],
        design=design)

    return (positions[-1,0] - x_init) / (2. * np.pi) 

### Optimization

In [ ]:
myoptimizer = sm.FlowBodyOptimizer(ROLLOUT, myobjective, optimizer=optax.adam(0.001))

opt_design = myoptimizer.run(
    init_design = mybody.design_defaults,
    n_steps     = 1001,
    print_every = 100,
    clip_min    = 0.01,
    clip_max    = 1.0,
    grad_clip   = 0.1,
    maximize    = True,    # True for x-component of velocity > 0
)

print("\nOPTIMUM DESIGN PARAMETERS:\n", mybody.design_variables)
print(opt_design)

### Figure

In [ ]:
positions, orientations, dofs = ROLLOUT.rollout(DT, N_STEPS, design=opt_design)
fig = make_figure(positions, orientations, dofs)
fig.show()